# Logic gates (OR / AND) on the SpikeEngine STDP FPGA

Adapted from this project's own `01_or_gate_tutorial.ipynb` / `02_and_gate_tutorial.ipynb` (`board_variants/npu_stdp_dev/notebooks/tutorials/`), which build these networks purely in software with `SNN.simulate()`. This notebook builds the SAME two networks and instead runs them on a real `spikeengine` board (or a software-only preview with `RUN_HARDWARE = False`), cross-checking every truth-table row against the software reference.

AND gate: given two inputs, fires only if BOTH inputs fire.
<br>OR gate: given two inputs, fires if AT LEAST ONE input fires.

Both networks are the simplest possible SpikeEngine topology: two input neurons (threshold 0, so any positive input current fires them immediately) synapsing onto a single output neuron. The gate's logic is entirely in the output neuron's threshold and synapse weights (SuperNeuroMAT's threshold compare is strict `>`, not `>=`, which is why these thresholds are one less than they'd otherwise look):
- **OR**: threshold 0, weight 1 on each input synapse -- either input alone (membrane 1 > 0) reaches threshold.
- **AND**: threshold 1, weight 1 on each input synapse -- both inputs together (membrane 2 > 1) are needed; one alone (membrane 1) does not exceed threshold 1.

In [1]:
import copy

import numpy as np

from superneuromat import SNN
from superneuromat import spikeengine as se

## Build the two networks

In [2]:
# stand-in for leak=inf (not representable in fixed-point); resets the membrane every tick
LEAK = 1000.0


def build_or_gate():
    net = SNN()
    ins = [net.create_neuron(threshold=0, leak=LEAK).idx for _ in range(2)]
    out = net.create_neuron(threshold=0, leak=LEAK).idx
    for i in ins:
        net.create_synapse(i, out, weight=1)
    return net, ins, out


def build_and_gate():
    net = SNN()
    ins = [net.create_neuron(threshold=0, leak=LEAK).idx for _ in range(2)]
    out = net.create_neuron(threshold=1, leak=LEAK).idx
    for i in ins:
        net.create_synapse(i, out, weight=1)
    return net, ins, out

## Configuration

Both networks are tiny (3 neurons, 2 synapses, no STDP, all-integer weights/thresholds), so `frac_bits=0` (plain integers, no fixed-point scaling) is enough. `weight_w`/`data_w` still need to match whichever board's bitstream is programmed -- `superneuromat.spikeengine.connect()` looks these up automatically from `superneuromat.spikeengine.boards.get_board(BOARD)`.

In [3]:
RUN_HARDWARE = True      # False -> software reference only
BOARD = 'basys3'      # 'basys3' | 'sp701' | 'zcu104'
PORT  = 'auto'           # 'auto' autodetects, or e.g. 'COM17'
FRAC_BITS = 0             # plain integers -- no fixed-point scaling needed for this example

In [4]:
dev = se.connect(port=PORT, board=BOARD) if RUN_HARDWARE else None
if dev is not None:
    dev.soft_reset()
    print('connected to', BOARD)

connected to basys3


## Run + verify a full truth table

For each of the 4 input combinations, `load_network()` is called once per gate (weights are fixed, no on-chip learning here), then each case does a `soft_reset()` (clears runtime state, config survives) + a 2-tick run: the input spike at t=0 fires the input neurons immediately (threshold 0), and their spike reaches the output neuron via the synapse on the NEXT tick (t=1) -- one-hop synaptic delay, the same model `SNN.simulate()` uses in software.

In [5]:
def software_truth_table(net, ins, out):
    rows = {}
    for x in (0, 1):
        for y in (0, 1):
            ref = copy.deepcopy(net)
            ref.add_spike(0, ins[0], x)
            ref.add_spike(0, ins[1], y)
            ref.simulate(2)
            rows[(x, y)] = bool(np.array(ref.spike_train)[-1][out])
    return rows


def hardware_truth_table(dev, net, ins, out, frac_bits):
    n_neurons = len(net.neuron_thresholds)
    rows = {}
    for x in (0, 1):
        for y in (0, 1):
            dev.soft_reset()
            se.load_network(dev, net, frac_bits=frac_bits)   # config survives soft_reset, but
                                                              # reload keeps this self-contained
            sched = {0: {ins[0]: x, ins[1]: y}}
            spikes = se.run_schedule(dev, sched, total_ticks=2, frac_bits=frac_bits,
                                     n_neurons=n_neurons)
            rows[(x, y)] = bool(spikes[-1][out])
    return rows


def run_gate(name, net, ins, out, expect_fn):
    sw = software_truth_table(net, ins, out)
    hw = hardware_truth_table(dev, net, ins, out, FRAC_BITS) if RUN_HARDWARE else None
    print(f'--- {name} gate ---')
    all_ok = True
    for x in (0, 1):
        for y in (0, 1):
            expect = expect_fn(x, y)
            sw_ok = sw[(x, y)] == expect
            line = f'  {x} {name.lower()} {y} = {int(sw[(x,y)])} (expect {int(expect)}, software {"OK" if sw_ok else "WRONG"})'
            if hw is not None:
                hw_ok = hw[(x, y)] == expect
                line += f', hardware {int(hw[(x,y)])} ({"OK" if hw_ok else "WRONG"})'
                all_ok = all_ok and hw_ok
            all_ok = all_ok and sw_ok
            print(line)
    print(f'{name} gate: {"ALL CASES PASS" if all_ok else "MISMATCH FOUND"}')
    return all_ok

## OR gate

In [6]:
or_net, or_ins, or_out = build_or_gate()
or_ok = run_gate('OR', or_net, or_ins, or_out, lambda x, y: bool(x or y))
assert or_ok

--- OR gate ---
  0 or 0 = 0 (expect 0, software OK), hardware 0 (OK)
  0 or 1 = 1 (expect 1, software OK), hardware 1 (OK)
  1 or 0 = 1 (expect 1, software OK), hardware 1 (OK)
  1 or 1 = 1 (expect 1, software OK), hardware 1 (OK)
OR gate: ALL CASES PASS


## AND gate

In [7]:
and_net, and_ins, and_out = build_and_gate()
and_ok = run_gate('AND', and_net, and_ins, and_out, lambda x, y: bool(x and y))
assert and_ok

--- AND gate ---
  0 and 0 = 0 (expect 0, software OK), hardware 0 (OK)
  0 and 1 = 0 (expect 0, software OK), hardware 0 (OK)
  1 and 0 = 0 (expect 0, software OK), hardware 0 (OK)
  1 and 1 = 1 (expect 1, software OK), hardware 1 (OK)
AND gate: ALL CASES PASS


## Cleanup

In [8]:
if dev is not None:
    dev.close()
    print('closed')

closed
